[Reference](https://medium.com/@rajeshmanikumarg/ollama-web-search-quick-and-simple-8a1f0c05ba04)

# Ollama Setup:
```
# To setup Ollama
# Linux
$ curl -fsSL https://ollama.com/install.sh | sh

# Windows
https://ollama.com/download/OllamaSetup.exe

# MAC
https://ollama.com/download/Ollama-darwin.zip

# Docker
https://hub.docker.com/r/ollama/ollama

# To download a model - you can use any model that supports tool
$ ollama run qwen3:latest
```

# Python Virtual Environment:
```
# Create a new Python Conda Environment python==3.13
pip install ollama
```

# Ollama API Key:
```
https://ollama.com/settings/keys
```

# Export to Set Key:
```
export OLLAMA_API_KEY="d6618d953d4d46d4b44c832e2d9a3e9b.5gBoQx1xxxxxxxxxxxxx"
```

# Web Search Agent with Tools:

In [1]:
import streamlit as st
from ollama import chat, web_fetch, web_search

# Initialize session state for messages
if "messages" not in st.session_state:
    st.session_state.messages = []

# App title
st.title("AI QA with Ollama's Web Search")
st.subheader("Ask any question and the get answer based on web searched data")

# Create a text input for the question
question = st.text_input("Enter your Question:", key="question_input")

# Create a button to submit the question
if st.button("Get Answer") or question:
    if question:
        # Add user message to session state
        st.session_state.messages.append({"role": "user", "content": question})

        # Display user message
        with st.chat_message("user"):
            st.markdown(question)

        # Initialize available tools
        available_tools = {'web_search': web_search, 'web_fetch': web_fetch}

        # Create a placeholder for the AI response
        with st.chat_message("assistant"):
            message_placeholder = st.empty()

            try:
                # Get the AI response (without thinking)
                response = chat(
                    model='qwen3:latest',
                    messages=st.session_state.messages,
                    tools=[web_search, web_fetch],
                    think=False
                )

                # Display content directly
                if response.message.content:
                    message_placeholder.markdown(response.message.content)

                # Add AI response to messages
                st.session_state.messages.append(response.message)

                # Handle tool calls and incorporate results into final response
                if response.message.tool_calls:
                    # Collect all tool results
                    tool_results = []
                    for tool_call in response.message.tool_calls:
                        function_to_call = available_tools.get(tool_call.function.name)
                        if function_to_call:
                            args = tool_call.function.arguments
                            result = function_to_call(**args)
                            tool_results.append(f"Source: {tool_call.function.name}\n\n{result}")

                    # Create enhanced prompt with tool results
                    if tool_results:
                        # Add tool results as a single user message to be processed by the model
                        enhanced_prompt = f"""
                        Based on the following information from web search:

                        {chr(10).join(tool_results)}

                        Please provide a comprehensive answer to this question: "{question}"
                        """

                        # Get final response with enhanced context
                        final_response = chat(
                            model='qwen3:latest',
                            messages=[{"role": "user", "content": enhanced_prompt}],
                            tools=[],
                            think=False
                        )

                        if final_response.message.content:
                            message_placeholder.markdown(final_response.message.content)

            except Exception as e:
                st.error(f"Error occurred: {str(e)}")

# Simple Python Raw WebSearch :

In [2]:
import ollama
response = ollama.web_search("When is Christmas in 2025?")
print(response)